# 太陽能板瑕疵檢測與成因說明

長榮大學資訊工程學系｜2026.12.05

無人機拍下太陽能板後，AI 能幫我們找出哪裡需要檢查嗎？

這次用 **case01 一起操作**，再從 **case02～case10 選一組自己試試**。

**看圖片 → PRB-FPN 找面板與瑕疵 → 整理成文字線索 → Qwen 說明可能原因**

| 模型 | 這次負責什麼？ |
|---|---|
| PRB-FPN 影像偵測模型 | 找出正常面板、破裂與熱點的位置 |
| Qwen 語言模型 | 根據偵測線索，產生一句繁體中文的可能成因 |

兩個模型都已由主辦方訓練好。這次直接拿來預測，這個動作稱為**推論**。
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andy6804tw/speech-archive/blob/main/2026/20261205-%E9%95%B7%E6%A6%AE%E5%A4%A7%E5%AD%B8%E8%B3%87%E5%B7%A5%E7%B3%BB-AI%E5%AF%A6%E4%BD%9C%E8%88%87%E5%AD%B8%E9%95%B7%E7%B6%93%E9%A9%97%E5%88%86%E4%BA%AB/notebooks/%E5%A4%AA%E9%99%BD%E8%83%BD%E6%9D%BF%E7%91%95%E7%96%B5%E6%AA%A2%E6%B8%AC/%E5%A4%AA%E9%99%BD%E8%83%BD%E6%9D%BF%E7%91%95%E7%96%B5%E6%AA%A2%E6%B8%AC%E8%88%87%E6%88%90%E5%9B%A0%E8%AA%AA%E6%98%8E.ipynb)


## 1. 開啟實作環境

1. 在 Colab 的「執行階段 → 變更執行階段類型」選擇 **T4 GPU**，或可用的 GPU。
2. 執行下方程式，自動從 GitHub 取得課程素材並準備好 10 組案例。
3. 再執行下一格，等待套件與官方模型準備完成。

第一次安裝與下載需要幾分鐘。模型解壓後約 **1.13 GB**，不需要 API 金鑰。
沒有 GPU 時也可以用 CPU 操作，速度會比較慢；重新啟動執行階段後，請從這裡開始。


In [ ]:
# @title 讀取實作素材
from pathlib import Path
from urllib.request import urlopen
import hashlib
import sys
import zipfile

素材檔 = Path("太陽能板實作素材.zip")
素材網址 = 'https://raw.githubusercontent.com/andy6804tw/speech-archive/main/2026/20261205-%E9%95%B7%E6%A6%AE%E5%A4%A7%E5%AD%B8%E8%B3%87%E5%B7%A5%E7%B3%BB-AI%E5%AF%A6%E4%BD%9C%E8%88%87%E5%AD%B8%E9%95%B7%E7%B6%93%E9%A9%97%E5%88%86%E4%BA%AB/notebooks/%E5%A4%AA%E9%99%BD%E8%83%BD%E6%9D%BF%E7%91%95%E7%96%B5%E6%AA%A2%E6%B8%AC/%E5%A4%AA%E9%99%BD%E8%83%BD%E6%9D%BF%E5%AF%A6%E4%BD%9C%E7%B4%A0%E6%9D%90.zip'
素材版本 = "1946a27c95fb4fb441a394e44543f07bbe66cba8e0974b9111e719c3918abf3c"

if not 素材檔.exists() or hashlib.sha256(素材檔.read_bytes()).hexdigest() != 素材版本:
    print("正在從 GitHub 讀取實作素材……")
    with urlopen(素材網址, timeout=60) as response:
        素材內容 = response.read()
    if hashlib.sha256(素材內容).hexdigest() != 素材版本:
        raise ValueError("素材下載不完整，請重新執行這格。")
    素材檔.write_bytes(素材內容)

教材目錄 = (Path.cwd() / "solar_classroom").resolve()
with zipfile.ZipFile(素材檔) as 壓縮檔:
    壓縮檔.extractall(教材目錄)
sys.path.insert(0, str(教材目錄))

from solar_classroom import SolarLab
lab = SolarLab(教材目錄)
print("素材已就緒：case01 示範，case02～case10 自選練習。")


In [ ]:
# @title 準備套件與官方模型（第一次需要等待）
lab.setup_environment()
lab.prepare_models()

## 2. case01：先用眼睛找一找

每組案例有兩張照片：**EO 是可見光影像**，看外觀；**IR 是熱影像**，看熱分布的差異。

先找找看：哪裡外觀不一樣？哪裡可能比較熱？

In [ ]:
lab.show_inputs("case01")

## 3. 讓 PRB-FPN 找出面板與瑕疵

執行後，看看模型畫出的框，和你剛才注意到的位置是否相同。

**藍框：正常面板　紅框：破裂（crack）　橘框：熱點（hotspot）**

框旁的數字是模型的信心分數，例如 `0.92`；分數高仍可能判斷錯誤。
程式會先依官方流程處理 IR，再進行偵測與結果整理。

In [ ]:
lab.detect_with_prb("case01")
lab.show_detections("case01")

## 4. 讓 Qwen 說明可能原因

PRB-FPN 負責「找到哪裡有什麼」，Qwen 接著根據這些線索說明原因。

先看一段**實際交給 Qwen 的文字**：它包含瑕疵類型、位置、數量，以及另一張影像的偵測結果。

In [ ]:
lab.show_prompt("case01")

這份 baseline 的 **Qwen 沒有直接看圖片**，它讀的是上面的文字。
使用的 Qwen 已用競賽成因標註微調過；預設會將生成結果對齊到候選句，讓句子完整。

現在執行下方程式，看看它提出什麼可能原因。**這是推測，仍需要查證。**

In [ ]:
lab.explain_with_qwen("case01")
lab.show_reasons("case01")

## 5. 換一組，你來試試

從下方選單挑 **一組**，有餘裕再多試另一組。

| 案例 | 可以觀察什麼？ |
|---|---|
| case02 | 正常面板對照 |
| case03 | 外觀破裂與熱影像的對照 |
| case04 | 密集面板中的小區域 |
| case05 | 破裂與熱點的對照 |
| case06 | 進階：條狀區域與可能漏掉的異常 |
| case07 | 進階：檢查熱點的成因文字 |
| case08 | 局部外觀破損 |
| case09 | 另一組外觀破損 |
| case10 | 另一組正常面板對照 |

選好後先執行這格看原圖，再執行下一格。換案例時，從選單這格開始往下重跑。

In [ ]:
# @title 選擇練習案例，先觀察原圖
CASE_ID = "case05" # @param ["case02", "case03", "case04", "case05", "case06", "case07", "case08", "case09", "case10"]
lab.show_inputs(CASE_ID)

In [ ]:
# PRB-FPN 先看圖片，Qwen 再讀偵測線索。
lab.detect_with_prb(CASE_ID)
lab.show_detections(CASE_ID)
lab.show_prompt(CASE_ID)
lab.explain_with_qwen(CASE_ID)
lab.show_reasons(CASE_ID)

## 6. 對照人工標註，留下你的觀察

模型沒有找到異常，不代表圖片一定正常；文字很通順，也不代表原因一定正確。
現在打開原始資料的人工標註，比較**位置、類別與成因文字**。

表格的數量只是幫助對照，**數量相同也可能框錯位置**。
人工標註可能沿著面板邊緣畫成多邊形，模型則會畫矩形框。

In [ ]:
lab.show_reference(CASE_ID)

### 我的練習紀錄

雙擊這個文字區塊，填入你的觀察：

- 我選擇的案例：
- 模型找到什麼？有沒有漏掉或多框的地方？
- Qwen 的說明合理嗎？你從哪張圖片或哪個線索判斷？

若這組沒有偵測到異常，就不會產生成因文字。可以再選一組異常案例比較。

## 7. 把練習結果存下來

最後輸出的 JSON 會記錄圖片名稱、物件類別、位置座標，以及異常物件的成因文字。
執行下方程式，下載**目前所選案例**的結果。

In [ ]:
結果檔 = lab.download_result(CASE_ID)

### 這次走過的流程

**圖片 → 偵測類別與位置 → 文字線索 → 可能成因 → 檢查結果**

這次用的是已訓練模型。若要讓模型學習新的影像特徵，才需要準備標註資料、訓練或微調，並用沒看過的資料驗證。

本次 10 組圖片選自競賽訓練集，模型已看過這份資料，適合練習操作；**不能當作模型在新資料上的準確率**。
下載檔也只包含你選的案例，並非完整競賽提交檔。

資料與模型：[競賽頁面](https://www.aidea-web.tw/topic/3c8ec48b-0dff-4b14-9b5e-d91bbb98c699?lang=zh)・[主辦方資料](https://drive.google.com/drive/folders/1PO9jYBlQXbkAgbp-xTySJWQdnBXF2wGz)